# Cross integration: Multiple RNA + ADT

Cross integration combines batches that all measure the same modalities. The task is to remove batch effects and keep the biological structure. This tutorial is for batches that each hold RNA and ADT. It runs StabMap and sciPENN on `D52mini`, 3,000 cells in three batches of the benchmark dataset `D52`, and then on data in your own format.

Methods run on Linux. On macOS or Windows, [open this notebook in Colab](https://colab.research.google.com/github/DSichang/scMultiBench/blob/main/notebooks/tutorial_cross_rna_adt.ipynb).

## 1. Install

This cell installs `multibench-sc`.

In [ ]:
import numpy, pandas
pins = f"numpy=={numpy.__version__} pandas=={pandas.__version__}"
%pip install -q "multibench-sc>=0.5.1" {pins}

## 2. Download the data and the environments

`mtb.data.fetch` downloads `D52mini` (2 MB) once. `mtb.env.install` downloads the environments for StabMap and sciPENN. The download is 1.4 GB on a computer without a GPU and 3.0 GB on one with a GPU.

In [ ]:
import multibench as mtb
import pandas as pd

pd.set_option("display.max_columns", None)   # show every column of the tables

mtb.data.fetch("D52mini")
METHODS = ["StabMap", "sciPENN"]
envs = mtb.env.install(METHODS)

## 3. Run the methods

`run` runs each method on `D52mini` and scores its output. The clustering metrics score how well the embedding's clusters and neighbourhoods match the cell types, and how separate the clusters are. The batch metrics measure how well the batches mix. The classification scores measure how well the cell types learned on one batch are predicted in the other batches. Higher is better for every metric.

<details>
<summary>Details</summary>

`params={"Method": {"key": value}}` sets a method's parameters, and `mtb.params_for` lists them. Many methods take none.

`mtb.load_run("out/D52mini")` reloads these results later without running anything.

</details>

In [ ]:
res = mtb.run("D52mini", "cross", methods=METHODS, out_dir="out/D52mini")
res.summary

## 4. Plot

`res.plot()` draws the scores as a bubble table. Circle size ranks the rows within each column and the colour scales each column, so a small gap can look large: `res.summary` has the values.

<details>
<summary>Details</summary>

Metrics are grouped by family: blue for clustering, green for batch correction, pink for classification. Each family starts with an `Overall` bar: each row's mean rank over the family's metrics, scaled so that the lowest gets the shortest bar.

A column with fewer than two different values is drawn grey. A figure of one method is grey everywhere.

</details>

In [ ]:
res.plot()

## 5. Your own data

Your data needs raw RNA and ADT counts in one `.h5ad` file, with a cell type and a batch for each cell.

The methods of this task read three batches, and UINMF also reads two.

Starting from something else? See the recipes for [a 10x `.h5` file](https://dsichang.github.io/scMultiBench/tutorials/run/#recipe-10x-h5), [labels in a CSV file](https://dsichang.github.io/scMultiBench/tutorials/run/#recipe-labels-csv), [no labels](https://dsichang.github.io/scMultiBench/tutorials/run/#recipe-no-labels) and [another number of donors](https://dsichang.github.io/scMultiBench/tutorials/run/#recipe-donors).

<details>
<summary>Details: export</summary>

The names in quotes say where your file holds each part. `"X"` is the main matrix, `"obsm:protein"` the matrix `obsm["protein"]`, `"layer:counts"` the layer `layers["counts"]`, and `"obs:celltype"` the column `obs["celltype"]`. A wrong name after `obsm:`, `layer:` or `obs:` raises an error that lists the names the file has.

The example keeps the protein names in `adata.uns["protein_names"]`. A DataFrame in `adata.obsm["protein"]` brings its own names in its columns.

The batches are numbered in the sorted order of their names, read as text: donors `A`, `B` and `C` become batches 1, 2 and 3, and `10` comes before `2`.

For one AnnData per batch, call `export_dataset` once per batch with `batch_index=N` instead of `batch=`.

The folder stores each matrix dense, so drop the genes or peaks you do not need first.

</details>

So that the notebook runs as it is, the next cell writes a small example file, `my_file.h5ad`, from 60% of the cells of `D52mini`, with a batch column.

In [ ]:
mtb.data.write_example("D52mini")

Read your file. For your own data, change the file name marked `# <-`. On Colab, upload the file first: click the folder icon on the left, then the upload icon.

In [ ]:
import anndata as ad

adata = ad.read_h5ad("my_file.h5ad")   # <- your file, every batch in it
adata

Write the dataset folder. Change each name in quotes marked `# <-`. The printout above lists the names in your file. Write a name listed after `obs:` with that prefix, as `"obs:celltype"`, and one after `obsm:` as `"obsm:protein"`.

In [ ]:
NAME = "MYCROSS_RNA_ADT"   # <- a new name for your own data
mtb.io.export_dataset(adata, "mydata/" + NAME,
                      rna="X",                # <- where the RNA counts are:
                                              #    "X" or "layer:counts"
                      adt="obsm:protein",     # <- where the proteins (ADT) are
                      labels="obs:celltype",  # <- the cell-type column
                      batch="obs:batch",      # <- the batch or donor column
                      category="cross", overwrite=True)
mtb.scan("mydata/" + NAME, "cross", methods=METHODS)

In [ ]:
mine = mtb.run("mydata/" + NAME, "cross", methods=METHODS,
               out_dir="out/" + NAME)
mine.summary

In [ ]:
mine.plot()

## 6. Stored results

The package ships stored results for 7 methods on the full `D52`. `load_results` reads them and `mtb.plot.bubble` draws them, without running anything.

In [ ]:
long = mtb.load_results("cross", dataset="D52")
mtb.plot.bubble(long)

## 7. Every method of this task

10 methods run this task. [This notebook](https://colab.research.google.com/github/DSichang/scMultiBench/blob/main/notebooks/tutorial_cross_rna_adt_all.ipynb) runs them all. It downloads 8 envs, 17.1 GB on a computer without a GPU and 22.1 GB on one with a GPU.

## Troubleshooting

`res.summary[res.summary.status != "scored"]` lists each method that failed or was skipped. Its `reason` column ends with the method's error output.

<details>
<summary>Details</summary>

| symptom | fix |
|---|---|
| a method fails on a small dataset | `mtb.scan` names what the method needs from the data's size in its `caveat` column |
| a method times out | raise `timeout=` in `run` |
| batch metrics use the wrong batches | `res.rescore(batch=my_vector)` scores again without running the methods |

</details>

## Next steps

- The guides: [run](https://dsichang.github.io/scMultiBench/tutorials/run/), [evaluate](https://dsichang.github.io/scMultiBench/tutorials/evaluate/), [plot](https://dsichang.github.io/scMultiBench/tutorials/plot/) and [discover methods](https://dsichang.github.io/scMultiBench/tutorials/discover/).
- The [interactive explorer](https://shiny.maths.usyd.edu.au/scMultiBench/) has the full benchmark's rankings, with no install needed.